In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# DATA LOADING AND AGGREGATION

base_path = 'meta_scan_csvs/cleaned'
room_files = {
    'kitchen': [],
    'hallway': [],
    'lab': [],
    'blinds': [], # meeting room
}

for room in room_files.keys():
    room_path = os.path.join(base_path, room)
    if os.path.exists(room_path):
        files = [f for f in os.listdir(room_path) if f.endswith('.csv')]
        room_files[room] = [os.path.join(room_path, f) for f in files]

def aggregate_perfetto_scan_extended(filepath):
    """Aggregate a time series scan with extended statistics"""
    df = pd.read_csv(filepath)
    df = df.drop(columns=['Time (s)'], errors='ignore')
    df = df.replace(-1, np.nan)
    
    features = {}
    
    for col in df.columns:
        values = df[col].dropna()
        if len(values) > 0:
            features[f'{col}_mean'] = values.mean()
            features[f'{col}_std'] = values.std()
            features[f'{col}_min'] = values.min()
            features[f'{col}_max'] = values.max()
            features[f'{col}_median'] = values.median()
            features[f'{col}_range'] = values.max() - values.min()
            features[f'{col}_iqr'] = values.quantile(0.75) - values.quantile(0.25)
            features[f'{col}_p10'] = values.quantile(0.10)
            features[f'{col}_p25'] = values.quantile(0.25)
            features[f'{col}_p75'] = values.quantile(0.75)
            features[f'{col}_p90'] = values.quantile(0.90)
            features[f'{col}_skew'] = values.skew()
            features[f'{col}_kurtosis'] = values.kurtosis()
            
            if values.mean() != 0:
                features[f'{col}_cv'] = values.std() / abs(values.mean())
            else:
                features[f'{col}_cv'] = 0
            
            features[f'{col}_first'] = values.iloc[0]
            features[f'{col}_last'] = values.iloc[-1]
            features[f'{col}_diff_first_last'] = values.iloc[-1] - values.iloc[0]
            
            if len(values) > 1:
                x = np.arange(len(values))
                slope = np.polyfit(x, values.values, 1)[0]
                features[f'{col}_trend'] = slope
            else:
                features[f'{col}_trend'] = 0
            
            if len(values) >= 10:
                rolling_std = values.rolling(window=10).std().mean()
                features[f'{col}_rolling_std'] = rolling_std if not pd.isna(rolling_std) else 0
            else:
                features[f'{col}_rolling_std'] = 0
            
            if len(values) > 2:
                diff = np.diff(values.values)
                sign_changes = np.sum(np.diff(np.sign(diff)) != 0)
                features[f'{col}_num_changes'] = sign_changes
            else:
                features[f'{col}_num_changes'] = 0
    
    return features

aggregated_scans_extended = {}
for room, files in room_files.items():
    for filepath in files:
        filename = os.path.basename(filepath)
        aggregated_scans_extended[(room, filename)] = aggregate_perfetto_scan_extended(filepath)

print(f"Total scans: {len(aggregated_scans_extended)}")

# TOP 40 FEATURES

TOP_40_FEATURES = [
    'gpu_util_diff_first_last',
    'Avg Preemption Delay_std',
    'gpu_util_iqr',
    'Avg Preemption Delay_p75',
    'gpu_util_std',
    'Avg Preemption Delay_p90',
    'gpu_util_first',
    'Avg Preemption Delay_iqr',
    'Avg Preemption Delay_max',
    'Avg Preemption Delay_range',
    'gpu_util_cv',
    'gpu_util_range',
    'app_vss_mb_median',
    'Avg Preemption Delay_last',
    'app_rss_mb_median',
    'display_refresh_rate_num_changes',
    'display_refresh_rate_diff_first_last',
    'display_refresh_rate_first',
    'Avg Preemption Delay_mean',
    'gpu_util_last',
    'gpu_util_max',
    'app_vss_mb_p25',
    'app_rss_mb_p25',
    'app_vss_mb_mean',
    'application_layer_count_p75',
    'application_layer_count_iqr',
    'gpu_util_p90',
    'application_layer_count_mean',
    'gpu_util_min',
    'app_uss_mb_median',
    'gpu_util_p75',
    'app_uss_mb_p25',
    'app_gpu_ms_iqr',
    'app_vss_mb_iqr',
    'gpu_util_trend',
    'Avg Preemption Delay_diff_first_last',
    'gpu_util_p10',
    '% Vertex Fetch Stall_last',
    'Vertices Shaded / Second_iqr',
    'Vertices Shaded / Second_last',
]

Total scans: 75


In [3]:
# ABLATION 1: 40 features, 300 trees

# BUILD DATASET

perfetto_data_40 = []
for (room, filename), features in aggregated_scans_extended.items():
    row = {'room': room, 'filename': filename}
    for feat in TOP_40_FEATURES:
        row[feat] = features.get(feat, 0)
    perfetto_data_40.append(row)

perfetto_df_40 = pd.DataFrame(perfetto_data_40).fillna(0)

X = perfetto_df_40.drop(columns=['room', 'filename'])
y = perfetto_df_40['room']

# MODEL TRAINING AND EVALUATION

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9867
Std deviation: 0.0267

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      1.00      1.00        15
     hallway       1.00      1.00      1.00        20
     kitchen       1.00      0.95      0.98        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.99        75
   macro avg       0.99      0.99      0.99        75
weighted avg       0.99      0.99      0.99        75



In [5]:
# ABLATION 2: 40 features, 200 trees

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9867
Std deviation: 0.0267

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      1.00      1.00        15
     hallway       1.00      1.00      1.00        20
     kitchen       1.00      0.95      0.98        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.99        75
   macro avg       0.99      0.99      0.99        75
weighted avg       0.99      0.99      0.99        75



In [6]:
# ABLATION 3: 40 features, 100 trees

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9867
Std deviation: 0.0267

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      1.00      1.00        15
     hallway       1.00      1.00      1.00        20
     kitchen       1.00      0.95      0.98        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.99        75
   macro avg       0.99      0.99      0.99        75
weighted avg       0.99      0.99      0.99        75



In [7]:
# ABLATION 4: 40 features, 25 trees

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=25, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9600
Std deviation: 0.0327

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      0.87      0.93        15
     hallway       1.00      1.00      1.00        20
     kitchen       0.91      0.95      0.93        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.96        75
   macro avg       0.96      0.95      0.96        75
weighted avg       0.96      0.96      0.96        75



In [9]:
# ABLATION 5: 20 features, 300 trees

TOP_20_FEATURES = TOP_40_FEATURES[:20]

perfetto_data_20 = []
for (room, filename), features in aggregated_scans_extended.items():
    row = {'room': room, 'filename': filename}
    for feat in TOP_20_FEATURES:
        row[feat] = features.get(feat, 0)
    perfetto_data_20.append(row)

perfetto_df_20 = pd.DataFrame(perfetto_data_20).fillna(0)

X = perfetto_df_20.drop(columns=['room', 'filename'])
y = perfetto_df_20['room']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9733
Std deviation: 0.0327

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      0.93      0.97        15
     hallway       1.00      1.00      1.00        20
     kitchen       0.95      0.95      0.95        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.97        75
   macro avg       0.98      0.97      0.97        75
weighted avg       0.97      0.97      0.97        75



In [10]:
# ABLATION 5: 20 features, 400 trees

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9733
Std deviation: 0.0327

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      0.93      0.97        15
     hallway       1.00      1.00      1.00        20
     kitchen       0.95      0.95      0.95        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.97        75
   macro avg       0.98      0.97      0.97        75
weighted avg       0.97      0.97      0.97        75



In [11]:
# ABLATION 5: 20 features, 100 trees

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9733
Std deviation: 0.0327

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       1.00      0.93      0.97        15
     hallway       1.00      1.00      1.00        20
     kitchen       0.95      0.95      0.95        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.97        75
   macro avg       0.98      0.97      0.97        75
weighted avg       0.97      0.97      0.97        75



In [12]:
# ABLATION 5: 20 features, 25 trees

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=25, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(rf_model, X_scaled, y, cv=kfold, scoring='accuracy')

print("\n5-FOLD CROSS-VALIDATION")
print(f"Mean accuracy: {cv_scores.mean():.4f}")
print(f"Std deviation: {cv_scores.std():.4f}")

y_pred_cv = cross_val_predict(rf_model, X_scaled, y, cv=kfold)

print("\nCLASSIFICATION REPORT")
print(classification_report(y, y_pred_cv))


5-FOLD CROSS-VALIDATION
Mean accuracy: 0.9600
Std deviation: 0.0533

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      blinds       0.93      0.93      0.93        15
     hallway       1.00      1.00      1.00        20
     kitchen       0.95      0.90      0.93        21
         lab       0.95      1.00      0.97        19

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75



In [23]:
# Define feature categories
gpu_util_features = [
    'gpu_util_diff_first_last',
    'gpu_util_iqr',
    'gpu_util_std',
    'gpu_util_first',
    'gpu_util_cv',
    'gpu_util_range',
    'gpu_util_last',
    'gpu_util_max',
    'gpu_util_p90',
    'gpu_util_min',
    'gpu_util_p75',
    'gpu_util_trend',
    'gpu_util_p10',
]

preemption_delay_features = [
    'Avg Preemption Delay_std',
    'Avg Preemption Delay_p75',
    'Avg Preemption Delay_p90',
    'Avg Preemption Delay_iqr',
    'Avg Preemption Delay_max',
    'Avg Preemption Delay_range',
    'Avg Preemption Delay_last',
    'Avg Preemption Delay_mean',
    'Avg Preemption Delay_diff_first_last',
]

memory_features = [
    'app_vss_mb_median',
    'app_rss_mb_median',
    'app_vss_mb_p25',
    'app_rss_mb_p25',
    'app_vss_mb_mean',
    'app_uss_mb_median',
    'app_uss_mb_p25',
    'app_vss_mb_iqr',
    'app_gpu_ms_iqr',
]

display_features = [
    'display_refresh_rate_num_changes',
    'display_refresh_rate_diff_first_last',
    'display_refresh_rate_first',
]

application_features = [
    'application_layer_count_p75',
    'application_layer_count_iqr',
    'application_layer_count_mean',
]

rendering_features = [
    '% Vertex Fetch Stall_last',
    'Vertices Shaded / Second_iqr',
    'Vertices Shaded / Second_last',
]

In [24]:
from sklearn.model_selection import train_test_split

def calculate_metrics(y_true, y_pred):
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    metrics['blinds_accuracy'] = report['blinds']['recall']
    metrics['hallway_accuracy'] = report['hallway']['recall']
    metrics['kitchen_accuracy'] = report['kitchen']['recall']
    metrics['lab_accuracy'] = report['lab']['recall']
    return metrics

X = perfetto_df_40.drop(columns=['room', 'filename'])
y = perfetto_df_40['room']

def ablation_study_perfetto_by_category(X_full, y):
    results = {}
    
    X_gpu = X_full[gpu_util_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_gpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only GPU Util'] = calculate_metrics(y_test, y_pred)
    
    X_preemption = X_full[preemption_delay_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_preemption, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Preemption Delay'] = calculate_metrics(y_test, y_pred)
    
    X_memory = X_full[memory_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_memory, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Memory'] = calculate_metrics(y_test, y_pred)
    
    X_display = X_full[display_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_display, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Display'] = calculate_metrics(y_test, y_pred)
    
    X_application = X_full[application_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_application, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Application'] = calculate_metrics(y_test, y_pred)
    
    X_rendering = X_full[rendering_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_rendering, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['Only Rendering'] = calculate_metrics(y_test, y_pred)
    
    no_gpu_features = [f for f in X_full.columns if f not in gpu_util_features]
    X_no_gpu = X_full[no_gpu_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_gpu, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No GPU Util'] = calculate_metrics(y_test, y_pred)
    
    no_preemption_features = [f for f in X_full.columns if f not in preemption_delay_features]
    X_no_preemption = X_full[no_preemption_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_preemption, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Preemption Delay'] = calculate_metrics(y_test, y_pred)
    
    no_memory_features = [f for f in X_full.columns if f not in memory_features]
    X_no_memory = X_full[no_memory_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_no_memory, y, 
        test_size=0.25, 
        random_state=42,
        stratify=y
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    y_pred = rf_model.predict(X_test_scaled)
    results['No Memory'] = calculate_metrics(y_test, y_pred)
    
    return results

perfetto_ablation_results = ablation_study_perfetto_by_category(X, y)

perfetto_ablation_df = pd.DataFrame(perfetto_ablation_results).T
perfetto_ablation_df_sorted = perfetto_ablation_df.sort_values('accuracy', ascending=False)

perfetto_ablation_df_sorted

,accuracy,blinds_accuracy,hallway_accuracy,kitchen_accuracy,lab_accuracy
Only GPU Util,1.000000,1.00,1.0,1.0,1.0
No Preemption Delay,1.000000,1.00,1.0,1.0,1.0
No Memory,1.000000,1.00,1.0,1.0,1.0
No GPU Util,1.000000,1.00,1.0,1.0,1.0
Only Memory,0.947368,1.00,0.8,1.0,1.0
Only Preemption Delay,0.842105,0.75,1.0,0.6,1.0
Only Rendering,0.789474,0.75,0.8,0.8,0.8
Only Application,0.526316,0.75,0.2,0.8,0.4
Only Display,0.473684,1.00,1.0,0.0,0.0
